## **Data Preprocessing**

This notebook prepares a common dataset for develop a CNN for image classification. The original 64 × 64 OCTMNIST splits are combined and reassigned into a reproducible, approximately 70:15:15 split. Images with identical pixel content are kept in the same partition; exact duplicates are removed from validation and testing for evaluation.

### 1. Dataset Loading and Data Analysis
OCTMNIST contains four grayscale retinal OCT classes: CNV, DME, drusen, and normal. The supplied images already satisfy the 64 × 64 maximum resolution, so no resizing is needed.

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from medmnist import INFO

dataset_path = Path("../data/octmnist_64.npz")
data = np.load(dataset_path)

# Dataset metadata
info = INFO["octmnist"]
class_names = list(info["label"].values())
class_ids = list(info["label"].keys())

print("===== OCTMNIST Dataset Summary =====")
print(f"Number of classes : {len(class_names)}")
print(f"Class names       : {', '.join(class_names)}")

total = 0

print("Available keys:", data.files)

for split in ["train", "val", "test"]:
    images = data[f"{split}_images"]
    labels = data[f"{split}_labels"].flatten()
    total += len(images)

    print(f"\n{split.upper()} SET")
    print(f"Number of images : {len(images):,}")
    print(f"Image shape      : {images.shape[1:]}")
    print(f"Class counts     :")

    counts = np.bincount(labels, minlength=len(class_names))
    for name, count in zip(class_names, counts):
        print(f"  {name:<8}: {count:,}")

print(f"\nTotal images     : {total:,}")

===== OCTMNIST Dataset Summary =====
Number of classes : 4
Class names       : choroidal neovascularization, diabetic macular edema, drusen, normal
Available keys: ['train_images', 'train_labels', 'val_images', 'val_labels', 'test_images', 'test_labels']

TRAIN SET
Number of images : 97,477
Image shape      : (64, 64)
Class counts     :
  choroidal neovascularization: 33,484
  diabetic macular edema: 10,213
  drusen  : 7,754
  normal  : 46,026

VAL SET
Number of images : 10,832
Image shape      : (64, 64)
Class counts     :
  choroidal neovascularization: 3,721
  diabetic macular edema: 1,135
  drusen  : 862
  normal  : 5,114

TEST SET
Number of images : 1,000
Image shape      : (64, 64)
Class counts     :
  choroidal neovascularization: 250
  diabetic macular edema: 250
  drusen  : 250
  normal  : 250

Total images     : 109,309


### 2. Image Resolution

OCTMNIST images are provided at a resolution of 64 × 64 pixels, satisfying the maximum resolution requirement of the assignment.

No additional resizing is required. The images remain grayscale with a single input channel.

In [13]:
# Verify image resolution
image_shape = data["train_images"].shape[1:]
print("Image resolution:", image_shape)

assert image_shape == (64, 64)
print("Image resolution verified!")

Image resolution: (64, 64)
Image resolution verified!


### 3. Dataset Integrity and Leakage Verification

The dataset partitions are checked for overlapping source indices, identical images appearing across partitions, and conflicting labels among retained images.

All verification checks confirm zero source-index overlap and zero exact-image overlap across training, validation, and testing partitions.

These checks prevent exact-image leakage but do not guarantee patient-level independence or the absence of visually similar, nonidentical scans.

In [14]:
import numpy as np
import hashlib
from collections import defaultdict
import numpy as np

X_all = np.concatenate([
    data["train_images"],
    data["val_images"],
    data["test_images"]
], axis=0)

y_all = np.concatenate([
    data["train_labels"],
    data["val_labels"],
    data["test_labels"]
], axis=0).flatten()

print("All images:", X_all.shape)
print("All labels:", y_all.shape)

SEED = 42
rng = np.random.default_rng(SEED)

def image_hash(image):
    return hashlib.sha256(
        np.ascontiguousarray(image).tobytes()
    ).hexdigest()

# Group identical images using their original indices
hash_to_indices = defaultdict(list)

for idx in range(len(X_all)):
    hash_to_indices[image_hash(X_all[idx])].append(idx)

# Separate consistent and conflicting label groups
consistent_groups = []
excluded_indices = []

for indices in hash_to_indices.values():
    labels = np.unique(y_all[indices])

    if len(labels) == 1:
        consistent_groups.append(
            (int(labels[0]), np.array(indices, dtype=np.int64))
        )
    else:
        excluded_indices.extend(indices)

print("Excluded conflicting-label images:", len(excluded_indices))


All images: (109309, 64, 64)
All labels: (109309,)
Excluded conflicting-label images: 553


### 4. Duplicate Handling  and  Dataset Splitting

To prevent exact image leakage, images are grouped using SHA-256 hashes calculated from their pixel data.

The dataset contains 7,117 duplicate-image groups, including 269 groups with inconsistent class labels. Since these labels cannot be reliably resolved using the available data, all 553 images belonging to these conflicting groups are excluded.

The remaining 108,756 images are assigned to training, validation, and testing partitions using a reproducible, duplicate-aware, class-stratified allocation targeting a 70:15:15 ratio.

All identical images are assigned to the same partition. Random seed 42 is used for reproducibility.

In [15]:
# Allocate complete duplicate groups independently per class
train_indices = []
val_indices = []
test_indices = []

for class_id in range(4):
    groups = [
        indices for label, indices in consistent_groups
        if label == class_id
    ]

    # Randomize group order reproducibly
    rng.shuffle(groups)

    # Assign larger groups first, retaining randomized tie order
    groups.sort(key=len, reverse=True)

    total_class_images = sum(len(g) for g in groups)

    targets = np.array([
        0.70, 0.15, 0.15
    ]) * total_class_images

    counts = np.zeros(3, dtype=np.int64)
    allocations = [[], [], []]

    for group in groups:
        # Select partition with greatest relative deficit
        deficits = (targets - counts) / targets
        selected = int(np.argmax(deficits))

        allocations[selected].extend(group.tolist())
        counts[selected] += len(group)

    train_indices.extend(allocations[0])
    val_indices.extend(allocations[1])
    test_indices.extend(allocations[2])

# Shuffle image order within each partition
train_idx = rng.permutation(train_indices)
val_idx = rng.permutation(val_indices)
test_idx = rng.permutation(test_indices)

# Construct dataset partitions
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

print("\n===== NEW DATASET PARTITIONS =====")

total_usable = len(train_idx) + len(val_idx) + len(test_idx)

for name, indices in [
    ("Training", train_idx),
    ("Validation", val_idx),
    ("Testing", test_idx)
]:
    print(
        f"{name}: {len(indices):,} "
        f"({len(indices) / total_usable * 100:.2f}%)"
    )

print("\nUsable images:", total_usable)


===== NEW DATASET PARTITIONS =====
Training: 76,128 (70.00%)
Validation: 16,314 (15.00%)
Testing: 16,314 (15.00%)

Usable images: 108756


#### Duplicate aware stratified splitting

In [16]:
def unique_image_indices(indices):
    seen_hashes = set()
    unique_indices = []

    for idx in indices:
        h = image_hash(X_all[idx])

        if h not in seen_hashes:
            seen_hashes.add(h)
            unique_indices.append(idx)

    return np.array(unique_indices, dtype=np.int64)

# Create unique validation and testing partitions
val_eval_idx = unique_image_indices(val_idx)
test_eval_idx = unique_image_indices(test_idx)

print("===== UNIQUE EVALUATION PARTITIONS =====")
print("Training:", len(train_idx))
print("Validation:", len(val_eval_idx))
print("Testing:", len(test_eval_idx))

assert len(val_eval_idx) == 15170
assert len(test_eval_idx) == 15172

assert len(set(val_eval_idx)) == len(val_eval_idx)
assert len(set(test_eval_idx)) == len(test_eval_idx)

assert len({
    image_hash(X_all[i]) for i in val_eval_idx
}) == len(val_eval_idx)

assert len({
    image_hash(X_all[i]) for i in test_eval_idx
}) == len(test_eval_idx)

print("\nUnique evaluation indices verified successfully!")

===== UNIQUE EVALUATION PARTITIONS =====
Training: 76128
Validation: 15170
Testing: 15172

Unique evaluation indices verified successfully!


#### Remove duplicates from evaluation sets

In [17]:
train_hashes = {
    image_hash(X_all[i]) for i in train_idx
}

val_hashes = {
    image_hash(X_all[i]) for i in val_idx
}

test_hashes = {
    image_hash(X_all[i]) for i in test_idx
}

assert train_hashes.isdisjoint(val_hashes)
assert train_hashes.isdisjoint(test_hashes)
assert val_hashes.isdisjoint(test_hashes)

print("Cross-split leakage verification: PASSED")

Cross-split leakage verification: PASSED


### 5. Saving Reproducible Dataset Partitions

The training, validation, testing, deduplicated evaluation, and excluded-image indices are saved in a compressed NumPy archive.

The archive also records the random seed and the SHA-256 checksum of the original dataset file.

This ensures that future experiments can reuse identical partitions and verify compatibility with the original dataset.

The splitting procedure modifies the official OCTMNIST benchmark partitions. Therefore, results from this experiment should not be directly compared with results using the official benchmark protocol.

In [18]:
from pathlib import Path
import hashlib
import numpy as np

# Location for saved split indices
split_path = Path("../data/split_indices_seed42.npz")
split_path.parent.mkdir(parents=True, exist_ok=True)

# Calculate SHA256 checksum of the original dataset file
sha256 = hashlib.sha256()

with open(dataset_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

dataset_checksum = sha256.hexdigest()

# Save verified dataset partitions
np.savez_compressed(
    split_path,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    val_eval_idx=val_eval_idx,
    test_eval_idx=test_eval_idx,
    excluded_idx=np.array(excluded_indices, dtype=np.int64),
    seed=np.array(SEED),
    dataset_sha256=np.array(dataset_checksum)
)

print("Split indices saved successfully!")
print("Location:", split_path.resolve())
print("Dataset SHA256:", dataset_checksum)
saved = np.load(split_path)

assert np.array_equal(saved["train_idx"], train_idx)
assert np.array_equal(saved["val_idx"], val_idx)
assert np.array_equal(saved["test_idx"], test_idx)

assert np.array_equal(saved["val_eval_idx"], val_eval_idx)
assert np.array_equal(saved["test_eval_idx"], test_eval_idx)

assert np.array_equal(
    np.sort(saved["excluded_idx"]),
    np.sort(excluded_indices)
)

assert saved["seed"].item() == SEED
assert saved["dataset_sha256"].item() == dataset_checksum

print("\nAll saved dataset indices and metadata verified!")

# Build deduplicated evaluation datasets
X_val_eval = X_all[val_eval_idx]
y_val_eval = y_all[val_eval_idx]

X_test_eval = X_all[test_eval_idx]
y_test_eval = y_all[test_eval_idx]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val_eval.shape, y_val_eval.shape)
print("Testing:", X_test_eval.shape, y_test_eval.shape)

Split indices saved successfully!
Location: A:\Academics\Semester 05\EN3150 - Pattern Recognition\PR A3\Oasis-EN3150-A3\data\split_indices_seed42.npz
Dataset SHA256: 323f8868b25f81d0634dcb9febbe3df74b44518f01a1c6e5f0246be62dcd73c3

All saved dataset indices and metadata verified!
Training: (76128, 64, 64) (76128,)
Validation: (15170, 64, 64) (15170,)
Testing: (15172, 64, 64) (15172,)


### 7. TensorFlow Data Pipeline

A reusable TensorFlow tf.data pipeline is created for model training and evaluation.

The pipeline uses a batch size of 32, training-only shuffling, batch-based normalization, grayscale channel expansion, and prefetching.

The training partition contains 76,128 images, while the deduplicated validation and test partitions contain 15,170 and 15,172 images respectively.

These datasets form the common experimental basis for Model A, Model B, and the two selected lightweight pretrained architectures, with model-specific preprocessing applied separately when required.

In [19]:
import tensorflow as tf
import numpy as np

SEED = 42
BATCH_SIZE = 32
NUM_CLASSES = 4

tf.random.set_seed(SEED)

def make_dataset(images, labels, training=False):
    # Read images from existing NumPy arrays
    def generator():
        for image, label in zip(images, labels):
            yield image, np.int32(label)

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(64, 64), dtype=tf.uint8),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )

    if training:
        ds = ds.shuffle(
            buffer_size=2048,
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE)

    def preprocess(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.expand_dims(image, axis=-1)
        return image, label

    ds = ds.map(
        preprocess,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds.prefetch(1)


train_ds = make_dataset(X_train, y_train, training=True)

# Use unique images for validation and testing
val_ds = make_dataset(X_val_eval, y_val_eval)
test_ds = make_dataset(X_test_eval, y_test_eval)

print("TensorFlow datasets created successfully!")

TensorFlow datasets created successfully!


### 8. Verification of Data Preprocessing

In [20]:
print("===== FINAL DATASET VERIFICATION =====")

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val_eval))
print("Testing samples:", len(X_test_eval))

for name, ds in [
    ("Training", train_ds),
    ("Validation", val_ds),
    ("Testing", test_ds)
]:
    images, labels = next(iter(ds))

    assert images.shape[1:] == (64, 64, 1)
    assert images.dtype == tf.float32
    assert labels.dtype == tf.int32
    assert tf.reduce_min(images) >= 0.0
    assert tf.reduce_max(images) <= 1.0

    print(f"{name}: batch verified")

print("All TensorFlow datasets verified!")

===== FINAL DATASET VERIFICATION =====
Training samples: 76128
Validation samples: 15170
Testing samples: 15172
Training: batch verified
Validation: batch verified
Testing: batch verified
All TensorFlow datasets verified!


In [21]:
print("===== FINAL CLASS DISTRIBUTION =====")

for name, labels in [
    ("Training", y_train),
    ("Validation", y_val_eval),
    ("Testing", y_test_eval)
]:
    counts = np.bincount(labels, minlength=4)

    print(f"\n{name}:")
    for class_id, count in enumerate(counts):
        print(
            f"  {info['label'][str(class_id)]}: "
            f"{count:,} ({count / len(labels) * 100:.2f}%)"
        )

===== FINAL CLASS DISTRIBUTION =====

Training:
  choroidal neovascularization: 26,073 (34.25%)
  diabetic macular edema: 8,094 (10.63%)
  drusen: 6,032 (7.92%)
  normal: 35,929 (47.20%)

Validation:
  choroidal neovascularization: 4,750 (31.31%)
  diabetic macular edema: 1,673 (11.03%)
  drusen: 1,201 (7.92%)
  normal: 7,546 (49.74%)

Testing:
  choroidal neovascularization: 4,751 (31.31%)
  diabetic macular edema: 1,674 (11.03%)
  drusen: 1,201 (7.92%)
  normal: 7,546 (49.74%)
